# 01 — De CSV a Parquet: ETL básico

El trabajo del 80% de los pipelines de datos del mundo:
leer crudo, transformar, escribir en formato columnar.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pathlib import Path

spark = (
    SparkSession.builder
    .appName("02_01_csv_a_parquet")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

RAW = "/home/jovyan/work/data/sample/ventas/ventas.csv"
OUT = "/home/jovyan/work/data/sample/ventas/parquet_limpio"

In [ ]:
schema = """
    id INT,
    fecha DATE,
    ciudad STRING,
    producto STRING,
    cantidad INT,
    precio DECIMAL(10, 2)
"""

raw = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")  # PERMISSIVE | DROPMALFORMED | FAILFAST
    .schema(schema)
    .csv(RAW)
)
raw.printSchema()
raw.show(3)

## Limpieza y transformaciones

In [ ]:
limpio = (
    raw
    # Filtrar filas inválidas
    .filter(F.col("cantidad") > 0)
    .filter(F.col("precio") > 0)
    # Normalizar strings
    .withColumn("ciudad", F.initcap(F.trim(F.col("ciudad"))))
    # Columna derivada
    .withColumn("ingreso", F.col("cantidad") * F.col("precio"))
    # Eliminar duplicados
    .dropDuplicates(["id"])
)

print("Filas después de limpieza:", limpio.count())

## Escritura a Parquet particionado

Particionamos por **año y mes** — baja cardinalidad, ideal para
filtros temporales.

In [ ]:
limpio_con_particiones = (
    limpio
    .withColumn("anio", F.year("fecha"))
    .withColumn("mes", F.month("fecha"))
)

(
    limpio_con_particiones
    .write
    .mode("overwrite")
    .partitionBy("anio", "mes")
    .parquet(OUT)
)
print("Escrito en", OUT)

## Verificación: partition pruning en acción

In [ ]:
# Sin filtro: lee todos los archivos
import time
t = time.time()
total = spark.read.parquet(OUT).count()
print(f"Sin filtro: {total} filas en {time.time() - t:.2f}s")

# Con filtro: Spark salta particiones que no matchean
t = time.time()
filtrado = spark.read.parquet(OUT).filter("anio = 2026 AND mes = 1").count()
print(f"anio=2026 AND mes=1: {filtrado} filas en {time.time() - t:.2f}s")

# Mirar la Spark UI: en el tab SQL, fijate que el nodo Filter pushdown
# muestra partition pruning

## ✅ Checkpoint

- [x] Leer CSV con schema explícito.
    
- [x] Limpiar, normalizar y deduplicar.
- [x] Escribir Parquet particionado por columnas de baja cardinalidad.
- [x] Verificar partition pruning.

**Siguiente**: `02_particionado.ipynb` — estrategia de particionado en
detalle.